# Minimum Sample Size Calculation

**Method:** Frequentist z-test for a single proportion (normal approximation to the binomial)

---

$$N = \frac{z^2 \cdot \hat{p} \cdot (1 - \hat{p})}{(\hat{p} - p_0)^2}$$

---

| Symbol | Meaning |
|--------|---------|
| $N$ | Minimum sample size (visitors) needed |
| $z$ | z-score for desired confidence level |
| $\hat{p}$ | Expected (true) conversion rate |
| $p_0$ | Target benchmark conversion rate |

### z-score by test type

| Test Type | $z$ | $\alpha$ | Use when... |
|-----------|-----|----------|-------------|
| **One-sided** | 1.65 | 0.05 (one tail) | You only need to detect CVR **above** target |
| **Two-sided** | 1.96 | 0.025 (each tail) | CVR could be **above or below** target |

In [1]:
import numpy as np
import plotly.graph_objects as go

def calculate_min_sample_size(target_cvr, expected_cvr, two_sided=False):
    """
    Calculates the minimum sample size needed to detect a difference
    between expected_cvr and target_cvr with 95% confidence.

    Statistical method: Frequentist z-test for a single proportion.
    Uses the normal approximation to the binomial distribution to estimate
    how many observations are needed to detect a meaningful difference
    between an observed proportion and a fixed benchmark.

    Parameters:
        target_cvr:  The benchmark / minimum viable conversion rate.
        expected_cvr: The conversion rate you believe is the true rate.
        two_sided:   False (default) → one-sided test (detect expected > target only)
                     True            → two-sided test (detect difference in either direction)
    """

    # 1. Validation Logic
    if expected_cvr == target_cvr:
        print(f"❌ ERROR: Expected CVR ({expected_cvr:.1%}) cannot equal Target ({target_cvr:.1%}). There must be a difference to detect.")
        return
    if two_sided is False and expected_cvr <= target_cvr:
        print(f"❌ ERROR: For a one-sided test, Expected CVR ({expected_cvr:.1%}) must be HIGHER than Target ({target_cvr:.1%}).")
        print(f"   ➡ Use two_sided=True if you want to detect a difference in either direction.")
        return

    # 2. Choose z-score based on test type
    if two_sided:
        z_score = 1.96   # 95% confidence, two-sided (α/2 = 0.025 each tail)
        test_label = "Two-Sided"
    else:
        z_score = 1.65   # 95% confidence, one-sided (α = 0.05 one tail)
        test_label = "One-Sided"

    # 3. The Formula: N = (z² × p × (1-p)) / (p - target)²
    z_score_squared = z_score ** 2
    numerator = z_score_squared * expected_cvr * (1 - expected_cvr)
    denominator = (expected_cvr - target_cvr) ** 2

    required_n = int(np.ceil(numerator / denominator))

    # 4. Print Results
    print(f"--- SAMPLE SIZE PLANNER ({test_label}) ---")
    print(f"Target Viability : {target_cvr:.1%}")
    print(f"Expected Reality : {expected_cvr:.1%}")
    print(f"Test Type        : {test_label} (z = {z_score})")
    print(f"-" * 45)
    print(f"You need roughly {required_n:,} visitors.")
    print(f"-" * 45)
    print()
    print(f"Why {test_label.lower()}?")
    if two_sided:
        print(f"  You want to detect whether the true CVR is significantly")
        print(f"  different from {target_cvr:.1%} in EITHER direction — it could")
        print(f"  be higher or lower. This is the safer choice when you're")
        print(f"  not sure which way the result will go.")
    else:
        print(f"  You only care about proving the true CVR is ABOVE {target_cvr:.1%}.")
        print(f"  This requires fewer visitors because you're only testing")
        print(f"  one direction, but it won't detect if CVR is below target.")

    # 5. Generate Data for the 'Planning Curve'
    offset_range = 0.10
    x_values = np.linspace(target_cvr + 0.002, target_cvr + offset_range, 500)
    y_values = (z_score_squared * x_values * (1 - x_values)) / ((x_values - target_cvr) ** 2)

    # For two-sided, also show the below-target curve
    if two_sided:
        x_below = np.linspace(max(0.001, target_cvr - offset_range), target_cvr - 0.002, 500)
        y_below = (z_score_squared * x_below * (1 - x_below)) / ((x_below - target_cvr) ** 2)

    # 6. Create Interactive Plotly Graph
    fig = go.Figure()

    primary_blue = '#1f77b4'
    below_orange = '#ff7f0e'

    # A. Above-target curve
    fig.add_trace(go.Scatter(
        x=x_values, y=y_values,
        mode='lines',
        name='Required Traffic (CVR > target)',
        line=dict(color=primary_blue, width=3)
    ))

    # B. Below-target curve (two-sided only)
    if two_sided:
        fig.add_trace(go.Scatter(
            x=x_below, y=y_below,
            mode='lines',
            name='Required Traffic (CVR < target)',
            line=dict(color=below_orange, width=3, dash='dash')
        ))

    # C. Your Specific Point
    fig.add_trace(go.Scatter(
        x=[expected_cvr], y=[required_n],
        mode='markers',
        name='Your Plan',
        marker=dict(color='red', size=12, symbol='diamond')
    ))

    # D. Target reference line
    fig.add_vline(
        x=target_cvr, line_dash="dot", line_color="gray",
        annotation_text=f"Target: {target_cvr:.1%}",
        annotation_position="top left"
    )

    # E. Layout
    fig.update_layout(
        title=dict(
            text=f"<b>Minimum Sample Size Planner ({test_label})</b><br>Target Viability: {target_cvr:.1%}  |  z = {z_score}",
            font=dict(size=20)
        ),
        xaxis_title="If your True Conversion Rate is...",
        yaxis_title="Visitors Needed (N)",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white",
        showlegend=True,
        plot_bgcolor='rgba(0,0,0,0)'
    )

    fig.add_annotation(
        x=expected_cvr, y=required_n,
        text=f"<b>{required_n:,} Visitors</b><br>needed at {expected_cvr:.1%}",
        ax=40, ay=-40,
        font=dict(size=12, color="red"),
        arrowcolor="red"
    )

    fig.show()


# ==========================================
# ENTER YOUR DATA HERE
# ==========================================
# Toggle: two_sided=False (one-sided) | two_sided=True (two-sided)

# TWO-SIDED: "CVR could be above or below 1%, detect either."
calculate_min_sample_size(target_cvr=0.01, expected_cvr=0.02, two_sided=True)

# ONE-SIDED: "I believe CVR is above 1%, prove it."
# calculate_min_sample_size(target_cvr=0.01, expected_cvr=0.02, two_sided=False)

--- SAMPLE SIZE PLANNER (Two-Sided) ---
Target Viability : 4.0%
Expected Reality : 3.0%
Test Type        : Two-Sided (z = 1.96)
---------------------------------------------
You need roughly 1,118 visitors.
---------------------------------------------

Why two-sided?
  You want to detect whether the true CVR is significantly
  different from 4.0% in EITHER direction — it could
  be higher or lower. This is the safer choice when you're
  not sure which way the result will go.
